# Notebook 05 — Evaluate + Generate Figures

Loads the trained checkpoint and:
1. Computes final metrics on the validation set (IoU, F1, precision, recall)
2. Generates a 4-panel prediction figure for LinkedIn/portfolio

**Environment**: Google Colab T4 (or CPU)  
**Runtime**: ~15 min

In [ ]:
# === Cell 0: Install dependencies (Colab only) ===
import subprocess, sys
try:
    import segmentation_models_pytorch
    print(f'smp {segmentation_models_pytorch.__version__} already installed')
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'segmentation-models-pytorch', 'albumentations'], check=True)
    print('Installed smp + albumentations')

In [ ]:
# === Cell 1: Imports ===
import json
import numpy as np
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm import tqdm

from google.colab import drive
drive.mount('/content/drive')

BASE       = Path('/content/drive/MyDrive/GeoAI/forest-disturbance')
LOCAL      = Path('/content/patches')
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PATCH_SIZE = 224
THRESHOLD  = 0.5

MEAN = np.array([338, 614, 659, 2600, 2200, 1320], dtype=np.float32)
STD  = np.array([250, 380, 500, 900,  850,  720],  dtype=np.float32)

print(f'Device: {DEVICE}')

In [ ]:
# === Cell 2: Copy patches to /content/ if needed ===
import subprocess, shutil, json

LOCAL = Path('/content/patches')

with open(BASE / 'data/patches/split.json') as f:
    SPLIT = json.load(f)

def _count(folder):
    return len(list(folder.glob('*.npy'))) if folder.exists() else 0

expected = len(SPLIT['train']) + len(SPLIT['val'])
actual   = _count(LOCAL / 'images_t1')

if actual < expected:
    print(f'{actual}/{expected} patches in /content/ — copying from Drive...')
    if LOCAL.exists():
        shutil.rmtree(LOCAL)
    LOCAL.mkdir(parents=True, exist_ok=True)
    for sub in ['images_t1', 'images_t2', 'masks']:
        subprocess.run(['cp', '-r', str(BASE / 'data/patches' / sub), str(LOCAL)], check=True)
        print(f'  {sub}: {_count(LOCAL / sub)} files')
    subprocess.run(['cp', str(BASE / 'data/patches/split.json'), str(LOCAL)], check=True)
else:
    print(f'Already complete: {actual} patches in /content/')

print(f'Val patches: {len(SPLIT["val"])}')

In [ ]:
# === Cell 3: Load model ===
import albumentations as A

model = smp.Unet(
    encoder_name='resnet34', encoder_weights=None,
    in_channels=12, classes=1, activation=None,
).to(DEVICE)

ckpt = BASE / 'models/best_forest_disturbance.pth'
assert ckpt.exists(), f'Checkpoint not found: {ckpt}'
model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
model.eval()
print(f'Loaded: {ckpt} ({ckpt.stat().st_size/1e6:.0f} MB)')


class ForestDataset(torch.utils.data.Dataset):
    def __init__(self, names, local_dir):
        self.names = names
        self.t1_dir  = local_dir / 'images_t1'
        self.t2_dir  = local_dir / 'images_t2'
        self.msk_dir = local_dir / 'masks'
        self.aug = A.CenterCrop(PATCH_SIZE, PATCH_SIZE)

    def __len__(self): return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]
        t1   = np.load(self.t1_dir  / name).astype(np.float32)
        t2   = np.load(self.t2_dir  / name).astype(np.float32)
        mask = np.load(self.msk_dir / name).astype(np.float32)
        t1   = (t1 - MEAN[:, None, None]) / (STD[:, None, None] + 1e-6)
        t2   = (t2 - MEAN[:, None, None]) / (STD[:, None, None] + 1e-6)
        img  = np.concatenate([t1, t2], axis=0).transpose(1, 2, 0)
        res  = self.aug(image=img, mask=mask)
        img  = res['image'].transpose(2, 0, 1)
        return torch.from_numpy(img), torch.from_numpy(res['mask']).unsqueeze(0), name

val_ds = ForestDataset(SPLIT['val'], LOCAL)
val_dl = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)

In [ ]:
# === Cell 4: Compute metrics on validation set ===
all_tp = all_fp = all_fn = 0.0
all_preds = []

with torch.no_grad():
    for imgs, masks, names in tqdm(val_dl, desc='Evaluating'):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        preds = model(imgs).sigmoid()
        binary = (preds > THRESHOLD).float()
        all_tp += (binary * masks).sum().item()
        all_fp += (binary * (1 - masks)).sum().item()
        all_fn += ((1 - binary) * masks).sum().item()
        # Store first batch for visualization
        if not all_preds:
            all_preds = [(imgs.cpu(), masks.cpu(), preds.cpu(), list(names))]

iou       = all_tp / (all_tp + all_fp + all_fn + 1e-6)
precision = all_tp / (all_tp + all_fp + 1e-6)
recall    = all_tp / (all_tp + all_fn + 1e-6)
f1        = 2 * precision * recall / (precision + recall + 1e-6)

metrics = {'IoU': round(iou, 4), 'F1': round(f1, 4),
           'Precision': round(precision, 4), 'Recall': round(recall, 4)}
print('\n=== Validation Metrics ===')
for k, v in metrics.items():
    print(f'  {k:12s}: {v:.4f}')

with open(BASE / 'results/evaluation.json', 'w') as f:
    json.dump(metrics, f, indent=2)

In [ ]:
# === Cell 5: Generate portfolio figure (5-panel) ===
# Pick the best-IoU patch from the first validation batch
imgs_b, masks_b, preds_b, names_b = all_preds[0]

# Find patch with highest IoU in this batch
best_i, best_iou_p = 0, 0.0
for i in range(imgs_b.shape[0]):
    p = (preds_b[i] > THRESHOLD).float()
    m = masks_b[i]
    tp = (p * m).sum(); union = p.sum() + m.sum() - tp
    iou_i = (tp / (union + 1e-6)).item()
    if iou_i > best_iou_p:
        best_iou_p, best_i = iou_i, i

raw_t1  = imgs_b[best_i].numpy()          # (12, H, W)
raw_t2  = imgs_b[best_i, 6:].numpy()       # T2 channels
gt_mask = masks_b[best_i, 0].numpy()
pred_pr = preds_b[best_i, 0].numpy()
pred_bn = (pred_pr > THRESHOLD).astype(np.float32)

def denorm_rgb(x6, ch_idx=[2, 1, 0]):
    bands = [(x6[c] * STD[c] + MEAN[c]) / 10000.0 for c in ch_idx]
    rgb = np.dstack(bands)
    lo, hi = np.percentile(rgb, [2, 98])
    return np.clip((rgb - lo) / (hi - lo + 1e-6), 0, 1)

rgb_t1 = denorm_rgb(raw_t1[:6])
rgb_t2 = denorm_rgb(raw_t1[6:])

# Error overlay
err = np.zeros((*pred_bn.shape, 3))
err[(pred_bn==1)&(gt_mask==1)] = [0.0, 0.82, 0.0]
err[(pred_bn==1)&(gt_mask==0)] = [1.0, 0.50, 0.0]
err[(pred_bn==0)&(gt_mask==1)] = [0.90, 0.10, 0.10]
err_alpha = np.where(err.sum(axis=2) > 0, 0.85, 0.0)

tp = int(((pred_bn==1)&(gt_mask==1)).sum())
fp = int(((pred_bn==1)&(gt_mask==0)).sum())
fn = int(((pred_bn==0)&(gt_mask==1)).sum())
patch_iou = tp / (tp + fp + fn + 1e-6)

import matplotlib.gridspec as gridspec
fig = plt.figure(figsize=(22, 6), facecolor='white')
outer = gridspec.GridSpec(2, 1, figure=fig, height_ratios=[1, 0.10], hspace=0.08,
                          left=0.02, right=0.98, top=0.90, bottom=0.02)
fig.suptitle(f'Forest Disturbance Detection — Gironde, France 2022  |  '
             f'Val IoU = {iou:.3f}  |  Patch IoU = {patch_iou:.3f}',
             fontsize=13, fontweight='bold', y=0.97)

inner = gridspec.GridSpecFromSubplotSpec(1, 5, subplot_spec=outer[0], wspace=0.04)
panels = [
    ('T1 — Pre-fire (July 2022)',   '#111111', rgb_t1, None),
    ('T2 — Post-fire (Sept 2022)',  '#111111', rgb_t2, None),
    ('Forest Loss Label',           '#c03030', rgb_t2, gt_mask),
    ('Prediction',                  '#2ea44f', (pred_pr > 0.3).astype(float), None),
    ('Prediction + Errors',         '#111111', None,   None),
]

for ci, (title, tc, data, mask) in enumerate(panels):
    ax = fig.add_subplot(inner[ci])
    ax.set_title(title, color=tc, fontsize=10, fontweight='bold', pad=5)
    ax.axis('off')
    if ci == 3:
        ax.imshow(data, cmap='Greens', vmin=0, vmax=1)
    elif ci == 4:
        ax.imshow(rgb_t2)
        ax.imshow(np.dstack([err, err_alpha]))
        ax.text(4, 14, f'IoU = {patch_iou:.3f}', color='white', fontsize=10,
                fontweight='bold', bbox={'fc': '#111111', 'alpha': 0.7, 'pad': 3})
        tp_p = mpatches.Patch(color=[0,.82,0],   label='True Positive')
        fp_p = mpatches.Patch(color=[1,.50,0],   label='False Positive')
        fn_p = mpatches.Patch(color=[.9,.1,.1],  label='False Negative')
        ax.legend(handles=[tp_p, fp_p, fn_p], loc='lower left', fontsize=7,
                  framealpha=0.8, handlelength=1.2, borderpad=0.5)
    elif mask is not None:
        ax.imshow(data)
        ov = np.zeros((*mask.shape, 4))
        ov[mask > 0] = [0.95, 0.25, 0.25, 0.85]
        ax.imshow(ov)
        ax.text(4, 210, f'{mask.mean()*100:.1f}% loss', fontsize=9,
                bbox={'fc': 'white', 'alpha': 0.75, 'pad': 3})
    else:
        ax.imshow(data)

ax_cap = fig.add_subplot(outer[1])
ax_cap.axis('off')
ax_cap.text(0.5, 0.5,
    f'Model: Early-Fusion U-Net (ResNet34, 12-channel input)  |  '
    f'Labels: Hansen GFC 2022 forest loss  |  '
    f'IoU={iou:.3f}  F1={f1:.3f}  Precision={precision:.3f}  Recall={recall:.3f}',
    ha='center', va='center', fontsize=9, color='#666666', transform=ax_cap.transAxes)

(BASE / 'results').mkdir(exist_ok=True)
out = BASE / 'results/prediction_demo.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out}')